In [ ]:
# TOKENIZER FROM SCRATCH & DATA PIPELINE

# =============================================================================
# UNDERSTANDING BPE — merges on a tiny example
# =============================================================================
# Short intro (you saw this idea in week1 tokenization; here we run several merges):
#
#   BPE = Byte Pair Encoding (subword tokenizer used by GPT-2/3/4-style models).
#
#   Problem with whole-word tokens:
#     "unhappiness" might be unknown / rare → model stuck (OOV).
#   BPE idea:
#     start from CHARACTERS, repeatedly MERGE the most common adjacent pair
#     into a new symbol ("e"+"s" → "es", "es"+"t" → "est", ...).
#     Rare words become a few known pieces; common words become one piece.
#
#   Training BPE = learn an ORDERED list of merges from corpus frequencies.
#   Using BPE later = apply those merges in order to new text → token ids.
#
# Tiny picture for this cell:
#   words + counts:  low×5, lower×2, newest×6, widest×3
#   start:           l o w ,  n e w e s t , ...
#   after merges:    low ,  ne w est , ...   (pieces like "est", "low")
#

import re, collections, json, os
import numpy as np


def get_stats(vocab):
    """Count how often each neighbor pair appears (weighted by word frequency)."""
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()          # "n e w e s t" → ['n','e','w','e','s','t']
        for i in range(len(symbols) - 1):
            # pair ('e','s') in a word with freq 6 → add 6, not 1
            pairs[symbols[i], symbols[i + 1]] += freq
    return pairs


def merge_vocab(pair, vocab):
    """Glue one bigram everywhere: ('e','s') turns 'e s' into 'es' in all words."""
    bigram = ' '.join(pair)       # ('e', 's') → 'e s'  (what we search for)
    replacement = ''.join(pair)   # ('e', 's') → 'es'   (new symbol)
    v_out = {}
    for word, freq in vocab.items():
        # string replace on the spaced spelling, e.g. "n e w e s t" → "n e w es t"
        v_out[word.replace(bigram, replacement)] = freq
    return v_out


# ---------------------------------------------------------------------------
# Toy corpus: whole words → spaced characters (each char starts as its own token)
# ---------------------------------------------------------------------------
raw = {'low': 5, 'lower': 2, 'newest': 6, 'widest': 3}
vocab_char = {' '.join(list(word)): freq for word, freq in raw.items()}
print("Initial word splits:", vocab_char)
# e.g. {'l o w': 5, 'l o w e r': 2, 'n e w e s t': 6, 'w i d e s t': 3}

# ---------------------------------------------------------------------------
# Run several BPE steps: count pairs → pick winner → merge → repeat
# ---------------------------------------------------------------------------
num_merges = 5
for i in range(num_merges):
    pairs = get_stats(vocab_char)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)   # most frequent adjacent pair
    print(f"Merge {i+1}: {best} (freq {pairs[best]})")
    vocab_char = merge_vocab(best, vocab_char)

print("After merges, word spellings:", vocab_char)
print("Final vocabulary pieces:", set(' '.join(vocab_char.keys()).split()))

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT (walkthrough of one run)
# ---------------------------------------------------------------------------
# Initial word splits:
#   {'l o w': 5, 'l o w e r': 2, 'n e w e s t': 6, 'w i d e s t': 3}
#   → each word is characters separated by spaces; number = corpus count.
#
# Merge 1: ('e', 's') (freq 9)
#   Why 9? newest has "e s" (×6) + widest has "e s" (×3) → 6+3=9.
#   Glue e+s → "es" everywhere those neighbors appear.
#
# Merge 2: ('es', 't') (freq 9)
#   After merge 1, both newest and widest have "es t" → merge to "est" (×9).
#   Sticky: "est" is now ONE token shared by newest/widest.
#
# Merge 3: ('l', 'o') (freq 7)
#   low×5 + lower×2 both start with "l o" → 7. Glue → "lo".
#
# Merge 4: ('lo', 'w') (freq 7)
#   "lo w" in low+lower → glue to "low" (still ×7 total across those words).
#
# Merge 5: ('n', 'e') (freq 6)
#   Only from newest×6 → "ne".
#
# After merges, word spellings:
#   {'low': 5, 'low e r': 2, 'ne w est': 6, 'w i d est': 3}
#   Read as: low is one piece; lower = low + e + r; newest = ne + w + est; etc.
#
# Final vocabulary pieces:
#   {'ne', 'r', 'w', 'e', 'i', 'd', 'low', 'est'}
#   = all symbols still used after 5 merges (new subwords + leftover chars).
#   Real GPT BPE: repeat thousands of times; saved merge list = the tokenizer.


Initial word splits: {'l o w': 5, 'l o w e r': 2, 'n e w e s t': 6, 'w i d e s t': 3}
Merge 1: ('e', 's') (freq 9)
Merge 2: ('es', 't') (freq 9)
Merge 3: ('l', 'o') (freq 7)
Merge 4: ('lo', 'w') (freq 7)
Merge 5: ('n', 'e') (freq 6)
After merges, word spellings: {'low': 5, 'low e r': 2, 'ne w est': 6, 'w i d est': 3}
Final vocabulary pieces: {'ne', 'r', 'w', 'e', 'i', 'd', 'low', 'est'}


In [4]:
# =============================================================================
# FULL BPE TOKENIZER CLASS — train merges, encode text → ids, decode ids → text
# =============================================================================
# Production-style skeleton (same ideas as GPT-2's BPE, just smaller / clearer):
#   train(text)  → learn which character pairs to glue, build vocab
#   encode(text) → apply those merges → list of integer token IDs
#   decode(ids)  → map IDs back → readable string
#
# Sticky picture for word "lowest":
#   start tokens:  l  o  w  e  s  t  </w>
#   after merges:  low  est</w>          (example — depends on learned merges)
#   encode:        [id(low), id(est</w>)]
#   decode:        glue tokens, turn </w> into a space
#
# </w> = end-of-word marker so "low" + "er" stays distinct from "lower" pieces.
#

class BPETokenizer:
    def __init__(self, vocab_size=1000):
        self.vocab_size = vocab_size           # target size (chars + merges)
        self.merges = {}    # (token_a, token_b) → merged_token string
        self.vocab = {}     # token string → integer id
        self.inv_vocab = {} # integer id → token string

    def _word_to_tokens(self, word):
        # "cat" → ['c', 'a', 't', '</w>']
        return list(word) + ['</w>']

    def _init_vocab_from_text(self, text):
        """Count words; seed vocab with every character (+ </w>)."""
        # words + leftover punctuation as separate tokens
        words = re.findall(r'\b\w+\b|\S', text.lower())
        word_freq = collections.Counter(words)

        char_set = set()
        for word, freq in word_freq.items():
            for ch in word:
                char_set.add(ch)
        char_set.add('</w>')

        # sorted → stable ids across runs
        self.vocab = {ch: i for i, ch in enumerate(sorted(char_set))}
        self.inv_vocab = {i: ch for ch, i in self.vocab.items()}
        return word_freq

    def train(self, text):
        """Learn BPE merges from corpus until vocab reaches vocab_size."""
        word_freq = self._init_vocab_from_text(text)

        # Spaced spelling of each unique word's current tokens
        #   "newest" → "n e w e s t </w>"
        splits = {word: ' '.join(self._word_to_tokens(word)) for word in word_freq.keys()}

        # How many NEW merge tokens we still need
        for merge_step in range(self.vocab_size - len(self.vocab)):
            # Count adjacent pairs across the corpus (weighted by word freq)
            pair_counts = collections.defaultdict(int)
            for word, freq in word_freq.items():
                symbols = splits[word].split()
                for i in range(len(symbols) - 1):
                    pair_counts[(symbols[i], symbols[i + 1])] += freq
            if not pair_counts:
                break

            best_pair = max(pair_counts, key=pair_counts.get)
            # New symbol name (toy rule: drop </w> from the right piece's spelling)
            # Keep </w> inside the merged piece so decode can restore spaces.
            # BUGFIX: old code stripped </w> → merges like ('s','</w>') became token "s"
            #         (collided with char "s") and decode glued words: "theapirate..."
            new_token = best_pair[0] + best_pair[1]

            # Register in vocab + remember this merge for encode()
            new_id = len(self.vocab)
            self.vocab[new_token] = new_id
            self.inv_vocab[new_id] = new_token
            self.merges[best_pair] = new_token

            # Apply merge in every word's spaced spelling
            #   "... e s ..." → "... es ..." when best_pair == ('e','s')
            for word in splits:
                splits[word] = splits[word].replace(' '.join(best_pair), new_token)

            if merge_step % 100 == 0:
                print(f"Merge {merge_step}: {best_pair} -> {new_token} (vocab size {len(self.vocab)})")

        print(f"Training complete. Final vocab size: {len(self.vocab)}")

    def encode(self, text):
        """Text → list of token ids (apply learned merges in training order)."""
        words = re.findall(r'\b\w+\b|\S', text.lower())
        ids = []
        for word in words:
            tokens = self._word_to_tokens(word)  # start from characters + </w>

            # Greedily merge: always apply the EARLIEST-learned merge that matches
            while True:
                pairs = [(tokens[i], tokens[i + 1]) for i in range(len(tokens) - 1)]
                merge_idx = float('inf')
                pair_to_merge = None
                for pair in pairs:
                    if pair in self.merges:
                        # earlier in self.merges.keys() = learned sooner = higher priority
                        idx = list(self.merges.keys()).index(pair)
                        if idx < merge_idx:
                            merge_idx = idx
                            pair_to_merge = pair
                if pair_to_merge is None:
                    break  # no more known merges apply

                new_token = self.merges[pair_to_merge]
                # Replace first occurrences of that pair left-to-right in this pass
                new_tokens = []
                i = 0
                while i < len(tokens):
                    if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair_to_merge:
                        new_tokens.append(new_token)
                        i += 2
                    else:
                        new_tokens.append(tokens[i])
                        i += 1
                tokens = new_tokens

            # Map each final piece to its id
            for token in tokens:
                ids.append(self.vocab.get(token, self.vocab.get('<UNK>', 0)))
        return ids

    def decode(self, ids):
        """Token ids → string (</w> becomes a space between words)."""
        tokens = [self.inv_vocab.get(i, '<UNK>') for i in ids]
        text = ''.join(tokens).replace('</w>', ' ')
        return text.strip()
# Next cell usually: train on a paragraph, encode a sentence, decode back.


In [ ]:
# =============================================================================
# TRAIN ON A TINY SALES/TECH CORPUS — then encode / decode a test sentence
# =============================================================================
# Flow:
#   1) Build a small repeated paragraph (toy stand-in for a big .txt file)
#   2) tokenizer.train(corpus)  → learn merges up toward vocab_size
#   3) encode(sentence) → list of ints   decode(ids) → string
#
# ---------------------------------------------------------------------------
# Corpus
# ---------------------------------------------------------------------------
# Multiplying by 100 does NOT add new words — it boosts COUNTS so pair
# frequencies are large and stable (same effect as seeing the paragraph
# many times in a bigger file).

corpus = """
Our product reduces operational costs by 30%. The API rate limit is 1000 requests per minute.
We offer 24/7 customer support. The integration supports REST and GraphQL endpoints.
Pricing starts at $500 per month. The deployment takes less than 5 minutes.
Security is SOC 2 compliant. What is your current pain point?
Let me show you the demo. The ROI is typically achieved in 3 months.
""" * 100

tokenizer = BPETokenizer(vocab_size=500)
tokenizer.train(corpus)

# ---------------------------------------------------------------------------
# Round-trip test
# ---------------------------------------------------------------------------
test_sentence = "The API rate limit is 500 requests per minute."
encoded = tokenizer.encode(test_sentence)
decoded = tokenizer.decode(encoded)
print("Original:", test_sentence)
print("Encoded:", encoded)
print("Decoded:", decoded)

# ---------------------------------------------------------------------------
# HOW TO READ A TYPICAL OUTPUT
# ---------------------------------------------------------------------------
# Merge 0: ('s', '</w>') -> s</w>   (vocab size ~36)
#   First learned glue: letter "s" at end-of-word is very common ("costs",
#   "requests", "minutes", ...). Vocab starts ~35 chars + </w>, then +1.
#
# Merge 100: ('end', 'po') -> endpo   (vocab size ~120)
#   After many steps, multi-char pieces appear (here building toward
#   "endpoint"/"endpoints"). Print every 100th merge is just a progress log.
#
# Training complete. Final vocab size: 186  (even if target was 500)
#   Why stop early? This corpus is tiny + highly repetitive. Once most words
#   collapse into a few pieces, few NEW adjacent pairs remain → train exits.
#   Bigger/diverse text would keep merging closer to vocab_size=500.
#
# Encoded: [42, 89, 91, ...]
#   One integer per subword piece for the test sentence (ids into tokenizer.vocab).
#   Same id always means the same piece.
#
# Decoded looking like:  theapiratelimitis500requestsperminuteth
#   That happened with the OLD merge bug (stripping </w>), so word boundaries
#   were lost and pieces smashed together. AFTER re-running the fixed
#   BPETokenizer cell + this cell, decode should keep spaces, closer to:
#     "the api rate limit is 500 requests per minute"
#   (still lowercased; punctuation handling is whatever our regex kept.)
#
# Sticky lesson: BPE is only as good as merge rules + </w> bookkeeping.
#   Always check encode→decode round-trip on a sentence you care about.


Merge 0: ('s', '</w>') -> s (vocab size 36)
Merge 100: ('end', 'po') -> endpo (vocab size 120)
Training complete. Final vocab size: 186
Original: The API rate limit is 500 requests per minute.
Encoded: [42, 89, 91, 93, 45, 127, 97, 38, 41, 48, 99, 36]
Decoded: theapiratelimitis500requestsperminuteth


In [6]:
# Data Pipeline for Language Model Pre‑Training

# Using our tokenizer, we'll create a PyTorch Dataset that yields input‑target pairs for next‑token prediction 
# (exactly what your mini‑GPT needs).

import torch
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, text, tokenizer, block_size):
        self.block_size = block_size
        self.tokens = tokenizer.encode(text)

    def __len__(self):
        return len(self.tokens) - self.block_size

    def __getitem__(self, idx):
        chunk = self.tokens[idx:idx+self.block_size+1]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y

# Prepare dataset from a larger text file (e.g., sales transcripts)
# For now, use the same corpus
dataset = TextDataset(corpus, tokenizer, block_size=32)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)
print(f"Number of training samples: {len(dataset)}")
print("Sample batch shapes:")
x_batch, y_batch = next(iter(dataloader))
print(x_batch.shape, y_batch.shape)

Number of training samples: 9568
Sample batch shapes:
torch.Size([16, 32]) torch.Size([16, 32])


In [9]:
# =============================================================================
# INTEGRATE WITH MiniGPT — BPE vocab size + one sample forward pass
# =============================================================================
# Plug the trained tokenizer into the same MiniGPT architecture from notebook 7.
# Difference vs Day 7: inputs are BPE subword ids (vocab≈186 here), not characters.

from pathlib import Path
import sys

# Make week2/ importable (repo root or week2 as cwd)
_WEEK2 = Path.cwd() / "week2"
if _WEEK2.is_dir():
    sys.path.insert(0, str(_WEEK2.resolve()))
elif Path.cwd().name == "week2":
    sys.path.insert(0, str(Path.cwd().resolve()))
else:
    sys.path.insert(0, str(Path("week2").resolve()))

from mini_gpt import MiniGPT

# Head must score every BPE piece → width = len(tokenizer.vocab)
vocab_size = len(tokenizer.vocab)
block_size = 32  # same context window as TextDataset / pos_embed table

model = MiniGPT(
    vocab_size,
    embed_dim=64,    # channel width per token
    num_heads=4,     # 64/4 = 16 dims per attention head
    ff_dim=128,      # MLP hidden size inside each block
    num_layers=3,    # stack 3 TransformerBlocks
    block_size=block_size,
)

print(f"MiniGPT ready | vocab_size={vocab_size} | params={sum(p.numel() for p in model.parameters()):,}")
print("Sample forward:", model(x_batch[:2]).shape)  # use 2 windows from the dataloader

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT
# ---------------------------------------------------------------------------
# MiniGPT ready | vocab_size=186 | params=126,010
#   vocab_size=186  — number of BPE pieces after train() (may be < target 500
#                     on this tiny corpus). Embedding + LM head use this width.
#   params=126,010  — total trainable weights (embeds + attn/MLP blocks + head).
#                     Small on purpose for a teaching toy.
#
# Sample forward: torch.Size([2, 32, 186])
#   batch=2   — we passed x_batch[:2] (two training windows)
#   32        — block_size seats (one logit vector per seat)
#   186       — score for every vocab id = "next-token" distribution per seat
#
# Next: train with CrossEntropyLoss on (logits vs y_batch), same loop as nb7,
#       iterating `dataloader` instead of the char-level loader.


MiniGPT ready | vocab_size=186 | params=126,010
Sample forward: torch.Size([2, 32, 186])
